In [104]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [105]:
# Load the dataset from the CSV file into a pandas DataFrame
data = pd.read_csv("Churn_Modelling.csv")



In [106]:
## Preprocess the data
### Drop irrelevant columns
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)#axis=1 this mean column vise
data

## Encode categorical variables
# Create a LabelEncoder instance for the 'Gender' column
label_encoder_gender = LabelEncoder()

# Fit the encoder on the 'Gender' column and transform it into numeric values
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [107]:
# we can not apply label encoder  on the Geography column because there are three values so for that we are using OneHotEncoder

## Onehot encode 'Geography
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoder

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [108]:
# Get the names of the one-hot encoded features for the 'Geography' column
onehot_encoder_geo.get_feature_names_out(['Geography'])


array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [109]:
# Create a new DataFrame from the one-hot encoded 'Geography' values
# 'geo_encoder' contains the transformed array
# 'onehot_encoder_geo.get_feature_names_out()' gives proper column names
geo_encoded_df = pd.DataFrame(
    geo_encoder,
    columns = onehot_encoder_geo.get_feature_names_out(['Geography'])
)

# Display the one-hot encoded DataFrame
geo_encoded_df


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [110]:
## Combine one-hot encoded columns with the original data

# Remove the original 'Geography' column and add the new one-hot encoded columns
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

# Show the first few rows of the updated dataset
data.head()


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [111]:
## Save the encoders

# Save the label encoder for 'Gender' column to a file
# This allows you to reuse it later for transforming new data
with open('label_encoder_gender_salary.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

# Save the one-hot encoder for 'Geography' column to a file
# This allows consistent transformation for new data in the future
with open('onehot_encoder_geo_salary.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)


In [112]:
## Divide the dataset into independent and dependent features

# X contains all features except the target column 'Exited'
X = data.drop('EstimatedSalary', axis=1)

y = data['EstimatedSalary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
scaler


,copy,True
,with_mean,True
,with_std,True


In [113]:
with open('scaler_salary.pkl','wb') as file:
    pickle.dump(scaler,file)
# till now apde khali data ne thodo barabr karyo chhe like j data nakamo chhe ene delete karyo and data ne transform karyo not apply ann yet 


In [114]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [115]:
(X_train.shape[1],)#number of features(columns) after removal of target column

(12,)

In [116]:
## Build Our ANN Model

model = Sequential([
    # Hidden Layer 1: 64 neurons, ReLU activation, connected to input features
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  
    # input_shape=(X_train.shape[1],) #tells Keras how many inputs each neuron in the first layer should expect.
    
    # Hidden Layer 2: 32 neurons, ReLU activation
    Dense(32, activation='relu'),  
    
    # Output Layer: 1 neuron with Sigmoid activation for binary classification
    Dense(1, activation='linear')  
])


In [117]:
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_9 (Dense)             (None, 64)                832       
                                                                 
 dense_10 (Dense)            (None, 32)                2080      
                                                                 
 dense_11 (Dense)            (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [118]:

log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
# Creates a folder to save logs for TensorBoard.
# datetime.datetime.now().strftime("%Y%m%d-%H%M%S") adds a timestamp, so each run gets a unique folder.
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [119]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)#ANN training will fail. without this 
loss=tensorflow.keras.losses.BinaryCrossentropy()
loss

model.compile(
    optimizer=opt,
    loss='mse',                  # ← mean squared error for regression
    metrics=['mae']              # ← mean absolute error is more readable
)



In [120]:

from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)


### Train the model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100
250/250 [==============================] - 3s 8ms/step - loss: 9605349376.0000 - mae: 81763.0000 - val_loss: 3548195584.0000 - val_mae: 50960.7461
Epoch 2/100
250/250 [==============================] - 1s 4ms/step - loss: 3378113792.0000 - mae: 49925.3008 - val_loss: 3381047040.0000 - val_mae: 50234.2461
Epoch 3/100
250/250 [==============================] - 1s 5ms/step - loss: 3333715712.0000 - mae: 49715.5352 - val_loss: 3394003456.0000 - val_mae: 50324.8164
Epoch 4/100
250/250 [==============================] - 1s 4ms/step - loss: 3324536064.0000 - mae: 49670.2578 - val_loss: 3359111424.0000 - val_mae: 50107.8789
Epoch 5/100
250/250 [==============================] - 1s 6ms/step - loss: 3308260096.0000 - mae: 49524.4922 - val_loss: 3364813312.0000 - val_mae: 50174.5938
Epoch 6/100
250/250 [==============================] - 1s 6ms/step - loss: 3307922432.0000 - mae: 49559.6250 - val_loss: 3353989888.0000 - val_mae: 50081.7344
Epoch 7/100
250/250 [=========================

In [122]:
import pandas as pd

# Reconstruct X_test as a DataFrame in original scale
X_test_unscaled = scaler.inverse_transform(X_test)
X_test_df = pd.DataFrame(X_test_unscaled, columns=X.columns)

# Round one-hot Geography columns back to integers if present
geo_cols = [c for c in X.columns if c.startswith("Geography")]
if geo_cols:
    X_test_df[geo_cols] = X_test_df[geo_cols].round().astype(int)

# Add actual and predicted values
y_test_reset = y_test.reset_index(drop=True)
y_pred = model.predict(X_test).flatten()

X_test_df['Actual_EstimatedSalary'] = y_test_reset
X_test_df['Predicted_Value'] = y_pred
X_test_df['Error'] = X_test_df['Predicted_Value'] - X_test_df['Actual_EstimatedSalary']

# Display combined result
display(X_test_df.head(20))

63/63 [==============================] - 0s 2ms/step


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Geography_France,Geography_Germany,Geography_Spain,Actual_EstimatedSalary,Predicted_Value,Error
0,596.0,1.0,32.0,3.0,96709.07,2.0,0.0,0.0,0.0,0,1,0,41788.37,95872.093750,54083.723750
1,623.0,1.0,43.0,1.0,0.00,2.0,1.0,1.0,0.0,1,0,0,146379.30,95530.539062,-50848.760937
2,601.0,0.0,44.0,4.0,0.00,2.0,1.0,0.0,0.0,0,0,1,58561.31,94622.468750,36061.158750
3,506.0,1.0,59.0,8.0,119152.10,2.0,1.0,1.0,0.0,0,1,0,170679.74,98676.078125,-72003.661875
4,560.0,0.0,27.0,7.0,124995.98,1.0,1.0,1.0,0.0,0,0,1,114669.79,98283.968750,-16385.821250
5,790.0,1.0,37.0,8.0,0.00,2.0,1.0,1.0,0.0,0,0,1,149418.41,98870.617188,-50547.792813
6,439.0,0.0,32.0,3.0,138901.61,1.0,1.0,0.0,0.0,0,0,1,75685.97,102369.492188,26683.522187
7,597.0,0.0,22.0,6.0,101528.61,1.0,1.0,0.0,1.0,0,1,0,70529.00,96497.742188,25968.742188
8,678.0,0.0,40.0,4.0,113794.22,1.0,1.0,0.0,0.0,0,0,1,16618.76,95102.968750,78484.208750
9,464.0,0.0,42.0,3.0,85679.25,1.0,1.0,1.0,0.0,0,1,0,164104.74,95056.804688,-69047.935312
